# Combined Pipeline v2 — Baseline + DA-Fusion + CEMS + CHARMS (full)

Single Kaggle-ready notebook with GroupKFold CV (5 folds), per-fold OOF predictions,
and a Kaggle submission CSV. All method additions are gated by config flags.

## CHARMS implementation (full port from `kaggle_baseline_pipeline_charms.ipynb`)

When `USE_CHARMS=True` this notebook runs the **complete** CHARMS stack:

| Component | Description |
|---|---|
| **FTTransformer** | Encodes tabular attributes (NDVI, height, state, species) into per-attribute embeddings for OT alignment; also regresses biomass targets (tabular auxiliary loss) |
| **OT channel alignment** | K-Means clusters 384 image dimensions by co-activation; Earth-Mover-Distance aligns channel-similarity matrices to attribute-similarity matrices → transfer matrix T |
| **Li2t heads** | One linear head per tabular attribute; each head predicts its attribute from T-weighted image features (channel → attribute knowledge transfer) |

Full combined loss: `L = L_image + λ_tab · L_tabular + λ_li2t · Li2t`

Tabular data is used **only during training**; inference is image-only.

## Method toggles (ablation matrix)

| USE_DA_FUSION | USE_CEMS | USE_CHARMS | Configuration |
|---|---|---|---|
| False | False | False | Plain baseline |
| True | False | False | DA-Fusion only |
| False | True | False | CEMS only |
| False | False | True | Baseline + full CHARMS |
| True | True | True | Combined (full stack) |

## 1. Setup & Imports

In [ ]:
import os, math, random, time, json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

# POT (Python Optimal Transport) — install if missing
try:
    import ot
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "POT", "-q"])
    import ot

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration

All method toggles, smoke-test toggles, and hyperparameters live here.

In [ ]:
cfg = SimpleNamespace(
    # --- Method toggles (all True = combined run; all False = baseline) ---
    USE_DA_FUSION=True,
    USE_CEMS=True,
    USE_CHARMS=True,  # full CHARMS: FT-Transformer + OT alignment + Li2t

    # --- Dry-run controls ---
    SMOKE_TEST=False,
    NUM_EPOCHS=80,
    NUM_FOLDS=5,
    MAX_ANCHORS_PER_EPOCH=None,  # CEMS: None = no cap

    # --- Paths (Kaggle layout) ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    synthetic_dir="/kaggle/input/da-fusion-synthetic-biomass",
    synthetic_csv="/kaggle/input/da-fusion-synthetic-biomass/train_augmented.csv",
    output_dir="/kaggle/working",

    # --- Model ---
    input_dim=384, latent_dim=32, output_dim=5, dropout=0.1,

    # --- Training ---
    lr=3e-4, weight_decay=1e-3, batch_size=32, seed=42,

    # --- CEMS (ignored if USE_CEMS=False) ---
    sigma=1e-3, cems_method=1, initial_d=5, use_hessian=False,

    # --- CHARMS (ignored if USE_CHARMS=False) ---
    n_clusters=40,       # K-Means groups for channel-similarity OT alignment
    tab_embed_dim=32,    # FT-Transformer per-attribute embedding dimension
    ft_n_blocks=2,       # FT-Transformer number of transformer blocks
    li2t_weight=0.1,     # Weight of Li2t auxiliary loss
    tab_loss_weight=0.5, # Weight of tabular-model regression loss
    ot_update_freq=5,    # Recompute OT transfer matrix every N epochs
    ot_subsample=512,    # Max samples for OT cost-matrix computation

    # --- Real-image flip4x augmentation ---
    use_flip4x=True,
)

if cfg.SMOKE_TEST:
    cfg.NUM_EPOCHS = 1
    cfg.NUM_FOLDS  = 1
    print("[SMOKE_TEST] NUM_EPOCHS=1, NUM_FOLDS=1")

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("Flags:  USE_DA_FUSION=", cfg.USE_DA_FUSION,
      " USE_CEMS=", cfg.USE_CEMS,
      " USE_CHARMS=", cfg.USE_CHARMS)
print("NUM_EPOCHS:", cfg.NUM_EPOCHS, " NUM_FOLDS:", cfg.NUM_FOLDS)
for p, label in [(TRAIN_CSV, "train.csv"), (TEST_CSV, "test.csv"),
                  (TRAIN_IMG_DIR, "train dir"), (TEST_IMG_DIR, "test dir"),
                  (cfg.dino_weights_dir, "dino dir"),
                  (cfg.synthetic_dir, "synth dir"),
                  (cfg.synthetic_csv, "synth csv")]:
    print(f"  {label:>12}: {os.path.exists(p)}  ({p})")

## 3. Load DINOv2 (frozen backbone)

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

_dummy = torch.zeros(1, 3, 252, 504, device=device)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls

## 4. Load Training CSV

Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all_real          = df_wide[TARGETS].values.astype(np.float32)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all_real shape: {Y_all_real.shape}")

## 4b. CHARMS — Tabular Feature Preparation

Extract numerical and categorical attributes from `train.csv` aligned to the
pivoted image order. Used **only during training** (not at test time).

| Attribute | Type | Notes |
|---|---|---|
| `Pre_GSHH_NDVI` | numerical | z-score normalised |
| `Height_Ave_cm` | numerical | z-score normalised |
| `State` | categorical | Australian state, label-encoded |
| `Species_first` | categorical | first species token, label-encoded |

Base arrays are (N_orig, ...) and will be repeated ×4 at the end of cell 6
to align with the flip4x augmented feature arrays.

In [ ]:
TAB_NUM_COLS = ["Pre_GSHH_NDVI", "Height_Ave_cm"]

# One row per image_id, keeping tabular side-channel columns
df_tab = df_raw.drop_duplicates("image_id").copy()
df_tab["Species_first"] = df_tab["Species"].str.split("_").str[0].fillna("Unknown")
df_tab = df_tab.set_index("image_id")

# Align to df_wide row order
df_tab_aligned = df_tab.loc[df_wide["image_id"].values]

# Numerical: z-score normalise
tab_num_scaler = StandardScaler()
X_tab_num_base = tab_num_scaler.fit_transform(
    df_tab_aligned[TAB_NUM_COLS].fillna(0).values.astype(np.float32)
)  # (N_orig, 2)

# Categorical: label-encode
state_le = LabelEncoder()
X_state_base = state_le.fit_transform(
    df_tab_aligned["State"].fillna("Unknown").values
).astype(np.int64)

species_le = LabelEncoder()
X_species_base = species_le.fit_transform(
    df_tab_aligned["Species_first"].values
).astype(np.int64)

X_tab_cat_base = np.stack([X_state_base, X_species_base], axis=1)  # (N_orig, 2)

N_NUM_ATTRS   = len(TAB_NUM_COLS)                       # 2
N_CAT_CLASSES = [len(state_le.classes_),
                 len(species_le.classes_)]
N_CAT_ATTRS   = len(N_CAT_CLASSES)                      # 2
D_TAB         = N_NUM_ATTRS + N_CAT_ATTRS               # 4

print(f"Tabular attributes: {N_NUM_ATTRS} numerical, {N_CAT_ATTRS} categorical (D={D_TAB})")
print(f"  State cardinality  : {N_CAT_CLASSES[0]}")
print(f"  Species cardinality: {N_CAT_CLASSES[1]}")
print(f"X_tab_num_base: {X_tab_num_base.shape}  X_tab_cat_base: {X_tab_cat_base.shape}")
print("Note: flip4x alignment is applied at the end of cell 6.")

## 5. Feature Extraction — DINOv2 CLS Tokens

In [ ]:
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

_ORIENTATIONS = [
    lambda img: img,
    lambda img: TF.hflip(img),
    lambda img: TF.vflip(img),
    lambda img: TF.hflip(TF.vflip(img)),
]

def _extract_one(path, flip_fn=None):
    img = Image.open(path).convert("RGB")
    if flip_fn is not None:
        img = flip_fn(img)
    x = _dino_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out  = dino(pixel_values=x, interpolate_pos_encoding=True)
        feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
    return feat

def extract_features(image_paths, augment=False, log_prefix=""):
    """Return (N, 384) if augment=False, (N*4, 384) if augment=True."""
    feats = []
    for i, p in enumerate(image_paths):
        if augment:
            for flip_fn in _ORIENTATIONS:
                feats.append(_extract_one(p, flip_fn))
        else:
            feats.append(_extract_one(p))
        if (i + 1) % 50 == 0:
            print(f"  {log_prefix}{i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)

## 6. Extract Real Training Features

In [ ]:
real_paths = [os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg") for iid in train_image_ids_all]
N_real = len(real_paths)

if cfg.use_flip4x:
    print(f"Extracting real features with 4-orientation augmentation ({N_real} × 4) ...")
    X_real         = extract_features(real_paths, augment=True,  log_prefix="real ")
    Y_real         = np.repeat(Y_all_real, 4, axis=0)
    real_image_ids = np.repeat(train_image_ids_all, 4)
    real_group_ids = np.repeat(np.arange(N_real), 4)
else:
    print(f"Extracting real features (no flip) for {N_real} images ...")
    X_real         = extract_features(real_paths, augment=False, log_prefix="real ")
    Y_real         = Y_all_real
    real_image_ids = train_image_ids_all
    real_group_ids = np.arange(N_real)

print(f"X_real: {X_real.shape}  Y_real: {Y_real.shape}  groups: {len(np.unique(real_group_ids))}")

# Align tabular arrays with flip4x repetition
if cfg.use_flip4x:
    X_tab_num_all = np.repeat(X_tab_num_base, 4, axis=0)  # (N*4, 2)
    X_tab_cat_all = np.repeat(X_tab_cat_base, 4, axis=0)  # (N*4, 2)
else:
    X_tab_num_all = X_tab_num_base.copy()
    X_tab_cat_all = X_tab_cat_base.copy()

print(f"X_tab_num_all: {X_tab_num_all.shape}  X_tab_cat_all: {X_tab_cat_all.shape}")

## 7. Extract Test Features (once)

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids   = df_test_unique["image_id"].values
test_image_paths = [os.path.join(TEST_IMG_DIR, f"{iid}.jpg") for iid in test_image_ids]

print(f"Extracting features for {len(test_image_paths)} test images ...")
X_test = extract_features(test_image_paths, augment=False, log_prefix="test ")
print(f"X_test: {X_test.shape}")

## 8. DA-Fusion — Load Synthetic CSV & Extract Features (once)

In [ ]:
if cfg.USE_DA_FUSION:
    def _source_id_from_path(p):
        return Path(str(p)).stem

    def _resolve_synth_path(image_path_field):
        candidates = [
            os.path.join(cfg.synthetic_dir, image_path_field),
            os.path.join(cfg.synthetic_dir, os.path.basename(image_path_field)),
            os.path.join(cfg.synthetic_dir, Path(image_path_field).name),
        ]
        for c in candidates:
            if os.path.exists(c):
                return c
        raise FileNotFoundError(
            f"Could not locate synthetic image: {image_path_field} under {cfg.synthetic_dir}"
        )

    df_synth_full = pd.read_csv(cfg.synthetic_csv)
    df_synth      = df_synth_full[df_synth_full["is_synthetic"].astype(bool)].copy()
    df_synth["source_id"] = df_synth["source_image"].apply(_source_id_from_path)
    print(f"Synthetic rows in CSV: {len(df_synth)}")

    synth_paths = [_resolve_synth_path(p) for p in df_synth["image_path"].values]
    print(f"Extracting DINOv2 features for {len(synth_paths)} synthetic images ...")
    X_synth_all = extract_features(synth_paths, augment=False, log_prefix="synth ")
    Y_synth_all = df_synth[TARGETS].values.astype(np.float32)
    synth_source_ids = df_synth["source_id"].values
    print(f"X_synth_all: {X_synth_all.shape}  Y_synth_all: {Y_synth_all.shape}")
else:
    X_synth_all      = np.zeros((0, cfg.input_dim), dtype=np.float32)
    Y_synth_all      = np.zeros((0, cfg.output_dim), dtype=np.float32)
    synth_source_ids = np.array([], dtype=object)
    print("USE_DA_FUSION=False — synthetic arrays empty.")

## 9. Model Architecture — BiomassModel

DINOv2 CLS → Encoder(384→128→32) → Head(32→32→5). Unchanged from baseline.

In [ ]:
class Encoder(nn.Module):
    """384 → 128 → 32 (GELU, Dropout)."""
    def __init__(self, input_dim=384, latent_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
        )
    def forward(self, x): return self.net(x)


class Head(nn.Module):
    """32 → 32 → 5 (GELU, Dropout)."""
    def __init__(self, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )
    def forward(self, z): return self.net(z)


class BiomassModel(nn.Module):
    """DINOv2 CLS → Encoder(384→128→32) → Head(32→32→5)."""
    def __init__(self, input_dim=384, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head    = Head(latent_dim, output_dim, dropout)

    def encode(self, x): return self.encoder(x)
    def forward(self, x): return self.head(self.encode(x))

    def forward_cems(self, args, x, y_scaled, get_batch_cems_fn):
        """CEMS augmentation path. Encode → detach → CEMS augment in 32-d."""
        latent_real = self.encode(x)
        latent_aug, y_aug_scaled = get_batch_cems_fn(
            args, latent_real.detach(), y_scaled, latent=True,
        )
        return self.head(latent_aug), y_aug_scaled


_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
print(f"BiomassModel params: {sum(p.numel() for p in _tmp.parameters()):,}")
del _tmp

## 9b. CHARMS Modules

Four components (only instantiated per fold when `USE_CHARMS=True`):

1. **`FTTransformer`** — encodes tabular attributes into per-attribute embeddings and regresses biomass targets.
2. **`Li2tHeads`** — one linear head per attribute; predicts each attribute from OT-weighted image channels.
3. **`compute_channel_clusters`** — K-Means on 384 feature-dimension co-activation vectors.
4. **`update_ot_transfer_matrix`** — builds T ∈ ℝ^{D×C} via Earth-Mover-Distance on channel/attribute similarity distributions.
5. **`CombinedDataset`** — dataset that carries tabular features alongside image features; marks real vs synthetic samples for selective Li2t loss.

In [ ]:
class FTTransformer(nn.Module):
    """
    Lightweight FT-Transformer for tabular data.

    Outputs:
      y_pred    : (B, output_dim)   regression prediction from CLS token
      attr_embs : (B, D, embed_dim) per-attribute embeddings (for OT alignment)
    """
    def __init__(self, n_num, cat_cardinalities, embed_dim=32, n_blocks=2,
                 output_dim=5, dropout=0.1):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_cardinalities)
        self.D = n_num + len(cat_cardinalities)

        self.num_tokenizers = nn.ModuleList([
            nn.Linear(1, embed_dim) for _ in range(n_num)
        ])
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(n_c + 1, embed_dim) for n_c in cat_cardinalities
        ])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=4 * embed_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.norm = nn.LayerNorm(embed_dim)
        self.reg_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, output_dim),
        )

    def forward(self, x_num, x_cat):
        tokens = []
        for i, tok in enumerate(self.num_tokenizers):
            tokens.append(tok(x_num[:, i:i+1]).unsqueeze(1))  # (B, 1, E)
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i]).unsqueeze(1))       # (B, 1, E)
        x   = torch.cat(tokens, dim=1)                         # (B, D, E)
        cls = self.cls_token.expand(x.shape[0], -1, -1)        # (B, 1, E)
        x   = self.norm(self.transformer(torch.cat([cls, x], dim=1)))
        return self.reg_head(x[:, 0, :]), x[:, 1:, :]          # y_pred, attr_embs


class Li2tHeads(nn.Module):
    """One prediction head per tabular attribute, operating on OT-weighted image channels."""
    def __init__(self, img_dim, n_num, cat_cardinalities):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_cardinalities)
        self.num_heads = nn.ModuleList([nn.Linear(img_dim, 1) for _ in range(n_num)])
        self.cat_heads = nn.ModuleList([
            nn.Linear(img_dim, n_c) for n_c in cat_cardinalities
        ])

    def compute_li2t_loss(self, img_feat, T, tab_num, tab_cat):
        """
        img_feat : (B, C)       raw image features (C=384)
        T        : (D, C)       OT transfer matrix
        tab_num  : (B, n_num)
        tab_cat  : (B, n_cat)
        """
        loss = img_feat.new_zeros(())
        for p in range(self.n_num):
            feat = img_feat * T[p].unsqueeze(0)
            pred = self.num_heads[p](feat).squeeze(-1)
            loss = loss + nn.functional.mse_loss(pred, tab_num[:, p])
        for q in range(self.n_cat):
            feat   = img_feat * T[self.n_num + q].unsqueeze(0)
            logits = self.cat_heads[q](feat)
            loss   = loss + nn.functional.cross_entropy(logits, tab_cat[:, q])
        return loss


def compute_channel_clusters(X_img: np.ndarray, n_clusters: int) -> np.ndarray:
    """K-Means on 384 feature-dimension co-activation vectors → (C,) cluster labels."""
    channel_vecs = X_img.T  # (C, N)
    norms = np.linalg.norm(channel_vecs, axis=1, keepdims=True) + 1e-8
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    return km.fit_predict(channel_vecs / norms).astype(np.int32)


def update_ot_transfer_matrix(
    X_img_np: np.ndarray,
    attr_embs_np: np.ndarray,
    cluster_labels: np.ndarray,
    n_clusters: int,
) -> np.ndarray:
    """
    Compute OT-based transfer matrix T ∈ R^{D × C}.

    Steps:
      1. For each cluster k: SI_k[i,j] = cosine-sim of cluster-k sub-vectors.
      2. For each attribute j: ST_j[i,j] = cosine-sim of FT-Transformer embeddings.
      3. Cost C[j,k] = ||SI_k − ST_j||²_F.
      4. Solve OT with uniform marginals → T_ot ∈ R^{D×k}.
      5. Map cluster weights back to individual dimensions → T_full ∈ R^{D×C}.
    """
    N, C = X_img_np.shape
    D    = attr_embs_np.shape[1]

    SI = np.empty((n_clusters, N, N), dtype=np.float32)
    for k in range(n_clusters):
        mask = cluster_labels == k
        if not mask.any():
            SI[k] = 0.0
            continue
        rep  = X_img_np[:, mask]
        nrm  = np.linalg.norm(rep, axis=1, keepdims=True) + 1e-8
        SI[k] = (rep / nrm) @ (rep / nrm).T

    ST = np.empty((D, N, N), dtype=np.float32)
    for j in range(D):
        rep  = attr_embs_np[:, j, :]
        nrm  = np.linalg.norm(rep, axis=1, keepdims=True) + 1e-8
        ST[j] = (rep / nrm) @ (rep / nrm).T

    C_mat = np.zeros((D, n_clusters), dtype=np.float64)
    for j in range(D):
        for k in range(n_clusters):
            diff        = SI[k] - ST[j]
            C_mat[j, k] = float(np.sum(diff * diff))
    C_mat /= (C_mat.max() + 1e-8)

    a    = np.ones(D,          dtype=np.float64) / D
    b    = np.ones(n_clusters, dtype=np.float64) / n_clusters
    T_ot = ot.emd(a, b, C_mat)  # (D, n_clusters)

    T_full = np.zeros((D, C), dtype=np.float32)
    for k in range(n_clusters):
        mask = cluster_labels == k
        if mask.any():
            T_full[:, mask] = T_ot[:, k:k+1].astype(np.float32)

    row_sums = T_full.sum(axis=1, keepdims=True) + 1e-8
    return T_full / row_sums  # row-normalised, (D, C)


class CombinedDataset(Dataset):
    """Returns (X, y, tab_num, tab_cat, is_real). Enables CHARMS loss on real samples only."""
    def __init__(self, X, y, tab_num, tab_cat, is_real):
        self.X       = torch.tensor(X,       dtype=torch.float32)
        self.y       = torch.tensor(y,       dtype=torch.float32)
        self.tab_num = torch.tensor(tab_num, dtype=torch.float32)
        self.tab_cat = torch.tensor(tab_cat, dtype=torch.long)
        self.is_real = torch.tensor(is_real, dtype=torch.bool)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i], self.tab_num[i], self.tab_cat[i], self.is_real[i]


print("CHARMS modules defined.")
if cfg.USE_CHARMS:
    _ft = FTTransformer(N_NUM_ATTRS, N_CAT_CLASSES, cfg.tab_embed_dim, cfg.ft_n_blocks,
                        cfg.output_dim, cfg.dropout)
    _li = Li2tHeads(cfg.input_dim, N_NUM_ATTRS, N_CAT_CLASSES)
    print(f"  FTTransformer params : {sum(p.numel() for p in _ft.parameters()):,}")
    print(f"  Li2tHeads params     : {sum(p.numel() for p in _li.parameters()):,}")
    del _ft, _li

## 10. CEMS — Inlined Algorithm

Only used when `USE_CEMS=True`. Operates in the joint `[latent_32d, y_scaled]` space.

In [ ]:
def _twonn_batch(zi):
    with torch.no_grad():
        dists = torch.cdist(zi.float(), zi.float())
        dists.fill_diagonal_(float('inf'))
        top2 = dists.topk(2, dim=1, largest=False).values
        r1, r2 = top2[:, 0], top2[:, 1]
        mask = r1 > 0
        if mask.sum() < 3:
            return 2
        mu = (r2 / r1)[mask]
        mu_sorted, _ = mu.sort()
        n       = len(mu_sorted)
        ecdf    = torch.arange(1, n + 1, dtype=torch.float32, device=zi.device) / n
        log_mu  = torch.log(mu_sorted)
        log_surv = -torch.log1p(-ecdf + 1e-10)
        denom   = (log_mu * log_mu).sum()
        if denom.abs() < 1e-12:
            return 2
        d = float((log_mu * log_surv).sum() / denom)
    if not math.isfinite(d) or d < 1:
        return 2
    return max(1, int(round(d)))


def _adjust_dims(x, y, xk=None, yk=None):
    if x.ndim > 2: x = x.reshape(x.shape[0], -1)
    if y.ndim == 1: y = y.reshape(y.shape[0], 1)
    m  = x.shape[-1]
    zi = torch.cat((x, y), dim=-1)
    zk = None
    if xk is not None and yk is not None:
        if xk.ndim > 3:
            xk = xk.reshape(xk.shape[0], xk.shape[1], -1)
        zk = torch.cat((xk, yk), dim=-1)
    return x, zi, zk, m


def _solve_ridge(a, b, lam=1.0):
    n   = a.shape[-1]
    eye = torch.eye(n, device=a.device, dtype=a.dtype)
    a_t = a.transpose(-2, -1)
    return torch.linalg.inv(a_t @ a + lam * eye) @ a_t @ b


def _get_projection(args, x, xk):
    x_c = x.transpose(-2, -1)
    x_c_mean = None
    if args.cems_method == 1:
        x_c_mean = torch.mean(x_c, -1)
        x_c = x_c - x_c_mean.unsqueeze(-1)
    else:
        assert xk is not None
        xk_t = xk.transpose(-1, -2)
        x    = x.unsqueeze(-1)
        x_c  = xk_t - x
    svd_input  = (x_c - x_c_mean.unsqueeze(-1) if x_c_mean is not None else x_c)
    svd_kwargs = {"driver": "gesvd"} if svd_input.is_cuda else {}
    basis, _, _ = torch.linalg.svd(svd_input, full_matrices=False, **svd_kwargs)
    u      = basis.transpose(-2, -1) @ x_c
    u_prev = u.transpose(-2, -1)
    if args.cems_method == 1:
        u_t  = u.transpose(-1, -2)
        u    = (u_t.unsqueeze(1) - u_t).transpose(-1, -2)
        n    = x.shape[0]
        mask = ~torch.eye(n, dtype=torch.bool, device=x.device)
        u    = -u.transpose(-1, -2)[mask].reshape(
            (u.shape[0], u.shape[2] - 1, u.shape[1])
        ).transpose(-1, -2)
    elif args.cems_method == 2:
        u = u.unsqueeze(0)
    return basis, u, u_prev, x_c_mean


def _estimate_grad_hessian(args, x, xk, d):
    tidx      = torch.triu_indices(d, d, device=x.device)
    ones_mult = torch.ones((d, d), device=x.device)
    ones_mult.fill_diagonal_(0.5)
    basis, u, u_prev, x_mean = _get_projection(args, x, xk)
    u_d = u[:, :d]
    f   = u[:, d:].transpose(-2, -1)
    if args.use_hessian:
        uu = torch.einsum('bki,bkj->bkij',
                           u_d.transpose(-2, -1), u_d.transpose(-2, -1))
        uu   = uu * ones_mult
        uu   = uu[:, :, tidx[0], tidx[1]].transpose(-2, -1)
        psi  = torch.cat((u_d, uu), dim=1).transpose(-2, -1)
    else:
        psi = u_d.transpose(-2, -1)
    lam    = torch.linalg.norm(psi, dim=(-1, -2)).mean()
    b_coef = _solve_ridge(psi, f, lam=lam).transpose(-2, -1)
    gradient = b_coef[..., :d]
    hessian  = torch.zeros((u.shape[0], b_coef.shape[1], d, d),
                            dtype=b_coef.dtype, device=b_coef.device)
    if args.use_hessian:
        hessian[..., tidx[0], tidx[1]] = b_coef[..., d:]
        hessian[..., tidx[1], tidx[0]] = b_coef[..., d:]
    return basis, gradient, hessian, u_d, u_prev, x_mean


def _sample_tangent(args, x, u_k_d, u_prev, x_mean, basis, grad, hess):
    d  = grad.shape[-1]
    nu = torch.distributions.Normal(0, args.sigma).sample(
        (x.shape[0], d, 1)
    ).to(x.device)
    f_nu = (grad @ nu).squeeze(-1)
    if args.use_hessian:
        nu_ex = nu.unsqueeze(1)
        f_nu  = f_nu + 0.5 * (nu_ex.transpose(-1, -2) @ hess @ nu_ex).squeeze((-1, -2))
    x_new_local = torch.cat((nu.squeeze(-1), f_nu), dim=-1)
    if args.cems_method == 1:
        x_new_local = x_new_local + u_prev
    x_cems = (basis @ x_new_local.unsqueeze(-1)).squeeze(-1)
    x_cems = x_cems + (x_mean if args.cems_method == 1 else x)
    return x_cems


def get_batch_cems(args, x, y, xk=None, yk=None, *, latent=True):
    x_shape, y_shape = x.shape, y.shape
    x_flat, zi, zk, m = _adjust_dims(x, y, xk, yk)
    d = args.id
    if latent:
        with torch.no_grad():
            d = _twonn_batch(zi)
    d = min(d, zi.shape[-1] - 1, zi.shape[-2] - 1)
    d = max(d, 1)
    basis, grad, hess, u_k_d, u_prev, x_mean = _estimate_grad_hessian(args, zi, zk, d)
    z_sampled = _sample_tangent(args, zi, u_k_d, u_prev, x_mean, basis, grad, hess)
    x_new = z_sampled[..., :m].reshape(x_shape)
    y_new = z_sampled[..., m:].reshape(y_shape)
    return x_new, y_new


def _next_power_of_2(n):
    if n < 1: return 1
    return 1 << (n - 1).bit_length()

def compute_neigh_size(d):
    base = d + d * (d + 1) // 2
    return _next_power_of_2(base + 1)

def precompute_knn(X, Y, neigh_size, compute_device="cpu"):
    XY = np.concatenate([X, Y], axis=1).astype(np.float32)
    t  = torch.tensor(XY, dtype=torch.float32)
    if compute_device in ("cuda", "mps"):
        try:
            t = t.to(compute_device)
        except Exception:
            pass
    dists = torch.cdist(t, t, p=2, compute_mode="donot_use_mm_for_euclid_dist")
    dists.fill_diagonal_(-float('inf'))
    sorted_idx = torch.argsort(dists, dim=1).cpu().numpy()
    return sorted_idx[:, :neigh_size].astype(np.int64)


print("CEMS functions defined.")

## 11. Loss, Metrics, Dataset, Eval

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)

def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()

def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)

def per_target_weighted_r2(y_true, y_pred):
    out = {}
    for i, t in enumerate(TARGETS):
        yt, yp = y_true[:, i], y_pred[:, i]
        ybar   = yt.mean()
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - ybar) ** 2) + 1e-12
        out[t] = float(1.0 - ss_res / ss_tot)
    return out

def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    return (
        total_loss / len(loader.dataset),
        weighted_global_r2(y_true, y_pred),
        rmse_per_target(y_true, y_pred),
        y_pred, y_true,
    )

print("Loss, metrics, dataset, eval_epoch defined.")

## 12. Per-Fold Training Function

Handles every toggle combination:
- `USE_DA_FUSION` → synthetic features appended to training set (per-fold leakage-filtered)
- `USE_CEMS` → anchor+kNN loop; real loss + augmented loss
- `USE_CHARMS` → FT-Transformer + OT alignment + Li2t loss on real samples

When both `USE_CEMS` and `USE_CHARMS` are active, CHARMS losses are computed on
the real-sample subset of each CEMS batch (synthetic samples have no tabular data).

In [ ]:
def train_one_fold(
    X_train_np, Y_train_np, X_val_np, Y_val_np, cfg, fold_idx,
    X_tab_num_tr=None, X_tab_cat_tr=None, N_real_in_train=None,
    verbose=True,
):
    """
    Train BiomassModel (+ CHARMS modules if enabled) on one fold.

    X_tab_num_tr    : (N_real_train, N_NUM_ATTRS)  — required when USE_CHARMS=True
    X_tab_cat_tr    : (N_real_train, N_CAT_ATTRS)  — required when USE_CHARMS=True
    N_real_in_train : int — first N rows in X_train_np are real (rest are DA-Fusion synthetic)
    """
    torch.manual_seed(cfg.seed + fold_idx)
    np.random.seed(cfg.seed + fold_idx)

    model = BiomassModel(
        cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout,
    ).to(device)

    # --- Instantiate CHARMS modules per fold ---
    if cfg.USE_CHARMS:
        assert X_tab_num_tr is not None and X_tab_cat_tr is not None, \
            "X_tab_num_tr / X_tab_cat_tr required when USE_CHARMS=True"
        assert N_real_in_train is not None

        ft_transformer = FTTransformer(
            n_num=N_NUM_ATTRS, cat_cardinalities=N_CAT_CLASSES,
            embed_dim=cfg.tab_embed_dim, n_blocks=cfg.ft_n_blocks,
            output_dim=cfg.output_dim, dropout=cfg.dropout,
        ).to(device)
        li2t_heads = Li2tHeads(
            img_dim=cfg.input_dim, n_num=N_NUM_ATTRS,
            cat_cardinalities=N_CAT_CLASSES,
        ).to(device)

        if verbose:
            print(f"    [CHARMS] K-Means channel clustering (k={cfg.n_clusters}) ...")
        channel_cluster_labels = compute_channel_clusters(
            X_train_np[:N_real_in_train], cfg.n_clusters
        )

        # Initialise transfer matrix: uniform weights
        T_mat = torch.ones(D_TAB, cfg.input_dim, device=device) / cfg.input_dim

        charms_params = list(ft_transformer.parameters()) + list(li2t_heads.parameters())
    else:
        charms_params = []

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + charms_params,
        lr=cfg.lr, weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(cfg.NUM_EPOCHS, 1), eta_min=cfg.lr / 100
    )

    val_ds     = BiomassDataset(X_val_np, Y_val_np)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

    y_scaler  = MinMaxScaler().fit(Y_train_np)
    Y_scaled  = y_scaler.transform(Y_train_np).astype(np.float32)

    neigh_size  = compute_neigh_size(cfg.initial_d) if cfg.USE_CEMS else 0
    knn_indices = (precompute_knn(X_train_np, Y_scaled, neigh_size, compute_device=device)
                   if cfg.USE_CEMS else None)

    cems_args = SimpleNamespace(
        sigma=cfg.sigma, cems_method=cfg.cems_method,
        id=cfg.initial_d, use_hessian=cfg.use_hessian,
    )

    X_t_full  = torch.tensor(X_train_np, dtype=torch.float32, device=device)
    Y_t_full  = torch.tensor(Y_train_np, dtype=torch.float32, device=device)
    Ys_t_full = torch.tensor(Y_scaled,   dtype=torch.float32, device=device)

    history = {"train_loss": [], "val_loss": [], "val_r2": []}
    wall_times = []
    best_val_r2 = -float("inf")
    best_state  = None

    mode  = ("CEMS" if cfg.USE_CEMS else "ERM")
    mode += " + DA-Fusion"   if cfg.USE_DA_FUSION else ""
    mode += " + CHARMS-full" if cfg.USE_CHARMS    else ""
    if verbose:
        print(f"    mode: {mode}")
        print(f"    N_train={len(X_train_np)}  N_val={len(X_val_np)}  epochs={cfg.NUM_EPOCHS}")

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        t_start = time.time()
        model.train()
        if cfg.USE_CHARMS:
            ft_transformer.train()
            li2t_heads.train()
        ep_loss, n_steps = 0.0, 0

        # --- OT transfer matrix refresh (CHARMS only) ---
        if cfg.USE_CHARMS and (epoch == 1 or epoch % cfg.ot_update_freq == 0):
            ft_transformer.eval()
            ot_n   = min(cfg.ot_subsample, N_real_in_train)
            ot_idx = np.random.choice(N_real_in_train, ot_n, replace=False)
            with torch.no_grad():
                t_num = torch.tensor(
                    X_tab_num_tr[ot_idx], dtype=torch.float32, device=device
                )
                t_cat = torch.tensor(
                    X_tab_cat_tr[ot_idx], dtype=torch.long, device=device
                )
                _, attr_embs = ft_transformer(t_num, t_cat)
                attr_embs_np = attr_embs.cpu().numpy()
            T_np  = update_ot_transfer_matrix(
                X_img_np       = X_train_np[ot_idx],
                attr_embs_np   = attr_embs_np,
                cluster_labels = channel_cluster_labels,
                n_clusters     = cfg.n_clusters,
            )
            T_mat = torch.tensor(T_np, dtype=torch.float32, device=device)
            ft_transformer.train()

        if cfg.USE_CEMS:
            # ---- Anchor+kNN loop ----
            N       = len(X_train_np)
            samples = np.random.permutation(N)
            if cfg.MAX_ANCHORS_PER_EPOCH is not None:
                samples = samples[:cfg.MAX_ANCHORS_PER_EPOCH]
            used = set()
            for anchor in samples:
                anchor = int(anchor)
                if anchor in used:
                    continue
                neigh_row  = knn_indices[anchor]
                candidates = neigh_row[1:]
                available  = [c for c in candidates if c not in used]
                n_neigh    = min(neigh_size - 1, len(available))
                if n_neigh == 0:
                    available = candidates[:neigh_size - 1].tolist()
                    n_neigh   = len(available)
                idx_all = np.concatenate(
                    [[anchor], np.array(available[:n_neigh], dtype=np.int64)]
                )
                X_b  = X_t_full[idx_all]
                Y_b  = Y_t_full[idx_all]
                Ys_b = Ys_t_full[idx_all]

                optimizer.zero_grad()

                loss_real = _weighted_smooth_l1(model(X_b), Y_b, _LOSS_WEIGHTS)

                pred_aug, ys_aug = model.forward_cems(cems_args, X_b, Ys_b, get_batch_cems)
                y_aug_np  = y_scaler.inverse_transform(
                    ys_aug.detach().cpu().numpy()
                ).astype(np.float32)
                y_aug_raw = torch.tensor(y_aug_np, dtype=torch.float32, device=device)
                loss_aug  = _weighted_smooth_l1(pred_aug, y_aug_raw, _LOSS_WEIGHTS)

                loss = loss_real + loss_aug

                # CHARMS: compute on real samples only (synthetics have no tabular data)
                if cfg.USE_CHARMS:
                    real_in_batch = idx_all[idx_all < N_real_in_train]
                    if len(real_in_batch) > 0:
                        X_real_b  = X_t_full[real_in_batch]
                        tab_num_b = torch.tensor(
                            X_tab_num_tr[real_in_batch], dtype=torch.float32, device=device
                        )
                        tab_cat_b = torch.tensor(
                            X_tab_cat_tr[real_in_batch], dtype=torch.long, device=device
                        )
                        tab_pred, _ = ft_transformer(tab_num_b, tab_cat_b)
                        Y_real_b    = Y_t_full[real_in_batch]
                        loss_tab    = _weighted_smooth_l1(tab_pred, Y_real_b, _LOSS_WEIGHTS)
                        loss_li2t   = li2t_heads.compute_li2t_loss(
                            X_real_b, T_mat, tab_num_b, tab_cat_b
                        )
                        loss = (loss
                                + cfg.tab_loss_weight * loss_tab
                                + cfg.li2t_weight     * loss_li2t)

                loss.backward()
                optimizer.step()
                ep_loss += loss_real.item() * len(idx_all)
                n_steps += len(idx_all)
                used.update(idx_all.tolist())

        else:
            # ---- Standard mini-batch loop ----
            if cfg.USE_CHARMS:
                # Build CombinedDataset — zero-pad tabular for synthetic rows
                n_synth = len(X_train_np) - N_real_in_train
                if n_synth > 0:
                    tab_num_full = np.vstack([
                        X_tab_num_tr,
                        np.zeros((n_synth, N_NUM_ATTRS), dtype=np.float32),
                    ])
                    tab_cat_full = np.vstack([
                        X_tab_cat_tr,
                        np.zeros((n_synth, N_CAT_ATTRS), dtype=np.int64),
                    ])
                else:
                    tab_num_full = X_tab_num_tr
                    tab_cat_full = X_tab_cat_tr

                is_real = np.zeros(len(X_train_np), dtype=bool)
                is_real[:N_real_in_train] = True

                train_ds = CombinedDataset(
                    X_train_np, Y_train_np, tab_num_full, tab_cat_full, is_real
                )
                train_loader = DataLoader(
                    train_ds, batch_size=cfg.batch_size,
                    shuffle=True, num_workers=0, drop_last=False,
                )
                for X_b, Y_b, tab_num_b, tab_cat_b, is_real_b in train_loader:
                    X_b, Y_b   = X_b.to(device), Y_b.to(device)
                    tab_num_b  = tab_num_b.to(device)
                    tab_cat_b  = tab_cat_b.to(device)
                    is_real_b  = is_real_b.to(device)

                    optimizer.zero_grad()
                    loss = _weighted_smooth_l1(model(X_b), Y_b, _LOSS_WEIGHTS)

                    if is_real_b.any():
                        X_real_b  = X_b[is_real_b]
                        tab_num_r = tab_num_b[is_real_b]
                        tab_cat_r = tab_cat_b[is_real_b]
                        Y_real_b  = Y_b[is_real_b]
                        tab_pred, _ = ft_transformer(tab_num_r, tab_cat_r)
                        loss_tab    = _weighted_smooth_l1(tab_pred, Y_real_b, _LOSS_WEIGHTS)
                        loss_li2t   = li2t_heads.compute_li2t_loss(
                            X_real_b, T_mat, tab_num_r, tab_cat_r
                        )
                        loss = (loss
                                + cfg.tab_loss_weight * loss_tab
                                + cfg.li2t_weight     * loss_li2t)

                    loss.backward()
                    optimizer.step()
                    ep_loss += loss.item() * len(X_b)
                    n_steps += len(X_b)
            else:
                train_ds = BiomassDataset(X_train_np, Y_train_np)
                train_loader = DataLoader(
                    train_ds, batch_size=cfg.batch_size,
                    shuffle=True, num_workers=0, drop_last=False,
                )
                for X_b, Y_b in train_loader:
                    X_b, Y_b = X_b.to(device), Y_b.to(device)
                    optimizer.zero_grad()
                    loss = _weighted_smooth_l1(model(X_b), Y_b, _LOSS_WEIGHTS)
                    loss.backward()
                    optimizer.step()
                    ep_loss += loss.item() * len(X_b)
                    n_steps += len(X_b)

        scheduler.step()

        tr = ep_loss / max(n_steps, 1)
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_loader, device)
        wall_times.append(time.time() - t_start)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)

        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == cfg.NUM_EPOCHS):
            print(f"      ep {epoch:3d}  tr={tr:.4f}  val_loss={val_loss:.4f}  "
                  f"val_R\u00b2={val_r2:.4f}  wall={wall_times[-1]:.1f}s")

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    _, val_r2_final, val_rmse_final, val_preds, val_true = eval_epoch(model, val_loader, device)
    return model, val_preds, val_true, history, wall_times, val_r2_final, val_rmse_final


print("train_one_fold defined.")

## 13. GroupKFold CV Loop — Run All Folds

In [ ]:
os.makedirs(cfg.output_dir, exist_ok=True)

gkf = GroupKFold(n_splits=5)  # fixed at 5 to preserve the baseline split
all_splits   = list(gkf.split(X_real, groups=real_group_ids))
active_splits = all_splits[:cfg.NUM_FOLDS]

oof_preds       = np.zeros_like(Y_real, dtype=np.float32)
oof_mask        = np.zeros(len(Y_real), dtype=bool)
fold_test_preds = []
fold_summaries  = []

SMOKE_LOG_TAG = "[SMOKE]" if cfg.SMOKE_TEST else ""
print(f"{SMOKE_LOG_TAG} Running {cfg.NUM_FOLDS} fold(s) with epochs={cfg.NUM_EPOCHS} "
      f"USE_DA_FUSION={cfg.USE_DA_FUSION} USE_CEMS={cfg.USE_CEMS} USE_CHARMS={cfg.USE_CHARMS}")

for fold_idx, (train_idx, val_idx) in enumerate(active_splits):
    print(f"\n===== Fold {fold_idx + 1}/{cfg.NUM_FOLDS} =====")

    # Leakage-safe split
    train_groups   = set(real_group_ids[train_idx].tolist())
    val_groups     = set(real_group_ids[val_idx].tolist())
    assert train_groups.isdisjoint(val_groups), "Group leakage in real split!"
    train_source_ids = set(real_image_ids[train_idx].tolist())

    X_tr_real = X_real[train_idx]; Y_tr_real = Y_real[train_idx]
    X_vl      = X_real[val_idx];   Y_vl      = Y_real[val_idx]

    # DA-Fusion: per-fold leakage-safe synthetic filter
    if cfg.USE_DA_FUSION and len(X_synth_all) > 0:
        synth_mask = np.array([sid in train_source_ids for sid in synth_source_ids])
        X_tr_synth = X_synth_all[synth_mask]
        Y_tr_synth = Y_synth_all[synth_mask]
        X_tr = np.vstack([X_tr_real, X_tr_synth]).astype(np.float32)
        Y_tr = np.vstack([Y_tr_real, Y_tr_synth]).astype(np.float32)
        kept_sources   = set(synth_source_ids[synth_mask].tolist())
        val_source_ids = set(real_image_ids[val_idx].tolist())
        assert kept_sources.isdisjoint(val_source_ids), \
            "DA-Fusion leakage: synthetic source is in val fold!"
        print(f"  real_train={len(X_tr_real)}  synth_kept={len(X_tr_synth)} "
              f"(of {len(X_synth_all)} total)  total_train={len(X_tr)}")
    else:
        X_tr, Y_tr = X_tr_real.astype(np.float32), Y_tr_real.astype(np.float32)
        print(f"  real_train={len(X_tr_real)}  (no synthetics)")

    N_real_in_train = len(X_tr_real)

    # CHARMS: extract tabular data for real training rows
    if cfg.USE_CHARMS:
        X_tab_num_tr = X_tab_num_all[train_idx]  # aligned with X_real[train_idx]
        X_tab_cat_tr = X_tab_cat_all[train_idx]
    else:
        X_tab_num_tr = None
        X_tab_cat_tr = None

    # Train
    model, val_preds, val_true, history, wall_times, val_r2_final, val_rmse = train_one_fold(
        X_tr, Y_tr, X_vl, Y_vl, cfg, fold_idx,
        X_tab_num_tr=X_tab_num_tr,
        X_tab_cat_tr=X_tab_cat_tr,
        N_real_in_train=N_real_in_train,
        verbose=True,
    )

    # Fold-0 wall-clock warning
    if fold_idx == 0 and len(wall_times) > 0:
        avg_wall = np.mean(wall_times)
        max_wall = np.max(wall_times)
        print(f"  [fold 0] epoch wall-clock — mean={avg_wall:.1f}s  max={max_wall:.1f}s")
        if max_wall > 600.0:
            print("  *** WARNING: fold-0 epoch exceeded 10 minutes. "
                  "Consider cfg.MAX_ANCHORS_PER_EPOCH or reducing synthetic ratio.")

    # OOF + test predictions
    oof_preds[val_idx] = val_preds
    oof_mask[val_idx]  = True

    model.eval()
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
    with torch.no_grad():
        tp = model(X_test_t).cpu().numpy()
    fold_test_preds.append(np.clip(tp, 0.0, None))

    # Per-fold reporting
    per_tgt_r2 = per_target_weighted_r2(val_true, val_preds)
    print(f"  >>> Fold {fold_idx} weighted R\u00b2 (aggregate): {val_r2_final:.4f}")
    for t in TARGETS:
        print(f"        {t:<16}: R\u00b2={per_tgt_r2[t]:+.4f}  RMSE={val_rmse[t]:.3f}")

    fold_summaries.append({
        "fold":              fold_idx,
        "val_r2":            float(val_r2_final),
        "per_target_r2":     per_tgt_r2,
        "per_target_rmse":   {k: float(v) for k, v in val_rmse.items()},
        "epoch_wall_times":  [float(x) for x in wall_times],
    })

print("\n===== CV complete =====")

## 14. Aggregate OOF Metrics & Persist to Disk

In [ ]:
if oof_mask.sum() > 0:
    oof_r2    = weighted_global_r2(Y_real[oof_mask], oof_preds[oof_mask])
    oof_per_t = per_target_weighted_r2(Y_real[oof_mask], oof_preds[oof_mask])
    oof_rmse  = rmse_per_target(Y_real[oof_mask], oof_preds[oof_mask])
    print(f"OOF weighted R\u00b2 (aggregate, {oof_mask.sum()} rows): {oof_r2:.4f}")
    for t in TARGETS:
        print(f"  {t:<16}: R\u00b2={oof_per_t[t]:+.4f}  RMSE={oof_rmse[t]:.3f}")
else:
    oof_r2 = float('nan')
    print("No OOF coverage.")

oof_out = os.path.join(cfg.output_dir, "oof_combined.npz")
np.savez(
    oof_out,
    oof_preds      = oof_preds,
    oof_mask       = oof_mask,
    y_true         = Y_real,
    real_image_ids = real_image_ids.astype(str),
    real_group_ids = real_group_ids,
    config         = np.array(json.dumps({
        "USE_DA_FUSION": cfg.USE_DA_FUSION,
        "USE_CEMS":      cfg.USE_CEMS,
        "USE_CHARMS":    cfg.USE_CHARMS,
        "NUM_EPOCHS":    cfg.NUM_EPOCHS,
        "NUM_FOLDS":     cfg.NUM_FOLDS,
        "SMOKE_TEST":    cfg.SMOKE_TEST,
        "oof_r2":        oof_r2,
    })),
)
print(f"OOF saved to: {oof_out}")

fold_json = os.path.join(cfg.output_dir, "fold_summaries.json")
with open(fold_json, "w") as f:
    json.dump({"oof_r2": oof_r2, "folds": fold_summaries}, f, indent=2)
print(f"Fold summaries saved to: {fold_json}")

## 15. Test Inference — Average Across Folds

In [ ]:
test_preds_mean = np.mean(np.stack(fold_test_preds, axis=0), axis=0)
test_preds_mean = np.clip(test_preds_mean, 0.0, None)
print(f"test_preds_mean: {test_preds_mean.shape}  range=[{test_preds_mean.min():.2f}, {test_preds_mean.max():.2f}]")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds_mean[i].round(2)))}")

## 16. Write Submission CSV — `submission_combined.csv`

In [ ]:
def prepare_submission(test_csv_path, predictions, image_ids):
    df_t = pd.read_csv(test_csv_path)
    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }
    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        return max(0.0, pred_dict.get(img_id, {}).get(target_name, 0.0))
    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]

submission = prepare_submission(TEST_CSV, test_preds_mean, test_image_ids)
out_path = os.path.join(cfg.output_dir, "submission_combined.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))